# 📘 Global Flood Monitoring & Alert Analysis

## Methodology

### 1. Setup & Imports
- Load necessary Python libraries for geospatial analysis, Dask distributed computing, and STAC API interaction.
- Configuration for the EODC environment is also set here.

### 2. Configuration & Authentication
- Set up connection to the Dask Gateway.
- Configure cluster options.
- Start the cluster.

### 3. Data Ingestion: CSV → Bounding Boxes (AOI)
- **Input:** CSV file with facility locations (`lat`, `lon`).
- **Standardization:** Column names are normalized, and coordinates are parsed.
- **AOI Creation:** A **square bounding box of 3km x 3km** is created for each point (extending 1500m in each cardinal direction using local AEQD projection). The result is transformed back to WGS84 (`EPSG:4326`).

### 4. STAC & Dask Processing
- **Source:** EODC STAC API (`GFM` collection).
- **Processing:**
    - Queries `ensemble_flood_extent`, `exclusion_mask`, and `ensemble_likelihood` bands.
    - **Flood Extent:** Aggregates time-series to find *any* flood occurrence (`max` over time).
    - **Exclusions:** Identifies permanent water/invalid pixels.
    - **Likelihood:** Calculates max likelihood of flood events.
- **Output:** GeoTIFF rasters stored in `outputs/flood`, `outputs/exclusion`, `outputs/likelihood`.

### 5. Raster Statistics Calculation
- **Projection:** Rasters are reprojected to an equal-area CRS (`EPSG:3035`) for accurate area measurement.
- **Metrics:**
    - Flood percentage within the AOI.
    - Validity of pixels (handling cloud cover/exclusions).
    - Centroid calculation.
- **Export:** `area_by_raster.csv` (raw stats) and `area_by_facility.csv` (aggregated stats).

### 6. Alert Matching (HEFAS)
- **Source:** HEFAS WFS layers (`NationalThresholdExceedance`, `PegelOnline`).
- **Logic:**
    - For each facility, checks for alerts within 100km radius on each day of the study period.
    - Computes distance to nearest alert and severity level.
- **Final Output:** `facility_with_floodstats_summary.csv` combining satellite flood detections with official alert history.


## 🛠️ 1. Setup & Imports
Load necessary Python libraries for geospatial analysis, Dask distributed computing, and STAC API interaction.
Configuration for the EODC environment is also set here.

In [2]:
import pyproj
import rioxarray as rxr
import xarray as xr
from datetime import datetime
from pystac_client import Client
from odc import stac as odc_stac
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, box as sbox
from shapely.ops import transform
from pyproj import CRS, Transformer
from __future__ import annotations
import os, re, time, warnings, traceback, socket, glob
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path
from rasterio.enums import Resampling
import requests
from requests.auth import HTTPBasicAuth
from io import BytesIO

from eodc_connect.dask import EODCDaskGateway


## ⚙️ 2. Configuration & Authentication
**User Action Required:** Enter your EODC password when prompted.
This initializes the connection to the Dask Gateway, allowing us to spawn a distributed cluster. Contact support@eodc.eu or tobias.stachl@eodc.eu 

In [3]:
your_username = "youremail@ext.ec.europa.eu"
gateway = EODCDaskGateway(username=your_username)

Enter your password: ········


### 2.1 Connect to Gateway
List available clusters associated with your user account. This helps verify that authentication was successful.

In [4]:
gateway.list_clusters()

[ClusterReport<name=dask-gateway.32d7c48ef6d6435faec46e8a34e4a9de, status=RUNNING>]

If you see gateway/s above, uncomment and run this block. It is necessary to keep only one cluster alive.

In [5]:
#cluster = gateway.connect(gateway.list_clusters()[0].name)
#cluster.shutdown()

### 2.2 Cluster Options & Startup

These are the default settings, and I recommend keeping them as they are.

Configure the Dask cluster specifications (cores, memory, docker image).
- **Workers:** scalable from 1 to 15 nodes.
- **Resources:** 8 cores, 16GB RAM per worker.
Finally, create the cluster and retrieve the dashboard link for monitoring.

In [6]:
# Define cluster options
cluster_options = gateway.cluster_options()

# Set the number of cores per worker
cluster_options.worker_cores = 8

# Set the memory per worker (in GB)
cluster_options.worker_memory = 16

# Specify the Docker image to use for the workers
cluster_options.image = "ghcr.io/eodcgmbh/cluster_image:2025.9.3"

# Create a new cluster with the specified options
cluster = gateway.new_cluster(cluster_options)

# Automatically scale the cluster between 1 and 10 workers based on workload
cluster.adapt(1, 15)  

# Optionally, scale the cluster to use only one worker
# cluster.scale(1)

# Get a Dask client for the cluster
client = cluster.get_client()
client.dashboard_link

'https://dask.services.eodc.eu/clusters/dask-gateway.70a59545c8a344b19d4dc3365603edc3/status'

2026-02-04 19:49:00,820 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


## 📍 3. Data Ingestion: CSV → Bounding Boxes (AOI)

1. **Load CSV:** Robust loading with encoding/separator autodetection.
2. **Normalize Columns:** Standardize `lat`/`lon` names.
3. **Validate:** Convert coordinates to numeric, drop empty rows.
4. **Geometries:** Create Points in WGS84 (`EPSG:4326`).
5. **Square Buffer:** Create a **3000m x 3000m square bounding box** (you can change it, if you want) centered on each point (extending 1500m in each cardinal direction). This uses a local AEQD projection for accurate metric dimensions before transforming back to WGS84.
6. **Store:** Save as `aoi_list_gdf`.
7. **Bounds:** Extract `minx, maxx, miny, maxy` for STAC querying.


In [ ]:
CSV_PATH = "flood_public_cos.csv"
HALF_SIZE_M = 1500

# robust data loading CSV with diacritics + separator autodetection
def read_csv_robust(path):
    encodings = ["utf-8-sig", "utf-8", "cp1250", "iso-8859-2", "latin1"]
    seps = [None, ";", "\t", ",", "|"]  # None = sniff (engine="python")
    last_err = None

    for enc in encodings:
        try:
            # first try separator autodetection
            df = pd.read_csv(path, sep=None, engine="python", encoding=enc, encoding_errors="replace")
            if df.shape[1] > 1:
                return df, enc, "sniffed"
            # if single column detected, try explicit separators
            for s in seps[1:]:
                try:
                    df2 = pd.read_csv(path, sep=s, encoding=enc, encoding_errors="replace")
                    if df2.shape[1] > 1:
                        return df2, enc, s
                except Exception as e:
                    last_err = e
        except Exception as e:
            last_err = e

    raise last_err if last_err else RuntimeError("CSV decode failed")

df, used_enc, used_sep = read_csv_robust(CSV_PATH)
print(f"Loaded CSV with encoding='{used_enc}', sep='{used_sep}'")


# If still only 1 column results, try fallback to ';' or TAB
if df.shape[1] == 1:
    if ";" in df.columns[0]:
        df = pd.read_csv(CSV_PATH, sep=";", encoding="utf-8-sig")
    elif "\t" in df.columns[0]:
        df = pd.read_csv(CSV_PATH, sep="\t", encoding="utf-8-sig")

# normalize column names
df.columns = [c.strip().lower() for c in df.columns]

# support aliases
if "lon" in df.columns and "longitude" not in df.columns:
    df = df.rename(columns={"lon": "longitude"})
if "lat" in df.columns and "latitude" not in df.columns:
    df = df.rename(columns={"lat": "latitude"})

required = {"facilityname", "latitude", "longitude"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV must contain columns: {sorted(required)}. Loaded columns: {list(df.columns)}")

# convert coordinates to numeric (if they were strings)
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
df = df.dropna(subset=["latitude", "longitude"])

# points in WGS84
points_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

# bbox function ±1500 m around point with local AEQD projection
def point_bbox_around(lat: float, lon: float, half_size_m: float = HALF_SIZE_M):
    wgs84 = CRS.from_epsg(4326)
    aeqd = CRS.from_proj4(f"+proj=aeqd +lat_0={lat} +lon_0={lon} +datum=WGS84 +units=m +no_defs")
    fwd = Transformer.from_crs(wgs84, aeqd, always_xy=True)
    inv = Transformer.from_crs(aeqd, wgs84, always_xy=True)
    x, y = fwd.transform(lon, lat)
    bbox_m = sbox(x - half_size_m, y - half_size_m, x + half_size_m, y + half_size_m)
    bbox_lonlat = transform(inv.transform, bbox_m)
    return bbox_lonlat

# create bboxes
aoi_list = points_gdf.apply(
    lambda r: point_bbox_around(float(r["latitude"]), float(r["longitude"]), HALF_SIZE_M),
    axis=1
)

aoi_list_gdf = gpd.GeoDataFrame(
    points_gdf.drop(columns="geometry").copy(),
    geometry=aoi_list,
    crs="EPSG:4326"
)

# Step 7: Bbox bounds to columns (for STAC)
bounds = aoi_list_gdf.bounds
aoi_list_gdf["minx"] = bounds["minx"]
aoi_list_gdf["miny"] = bounds["miny"]
aoi_list_gdf["maxx"] = bounds["maxx"]
aoi_list_gdf["maxy"] = bounds["maxy"]

n0 = len(df)  # original count
df = df.dropna(subset=["latitude", "longitude"])
n1 = len(df)

print(f"Originally in CSV: {n0} rows")
print(f"After dropping rows without coordinates: {n1}")
print(f"After creating bboxes: {len(aoi_list_gdf)}")

bboxes_gdf = aoi_list_gdf

# optionally check:
display(aoi_list_gdf.head())


Loaded CSV with encoding='utf-8-sig', sep='sniffed'
Originally in CSV: 1505 rows
After dropping rows without coordinates: 1505
After creating bboxes: 1505


,facilityname,countrycode,floodrisk,latitude,longitude,geometry,minx,miny,maxx,maxy
0,Panasonic Industrial Devices Materials Europe ...,AT,9.31,48.231083,14.496667,"POLYGON ((14.51685 48.21759, 14.51686 48.24457...",14.476471,48.217591,14.516863,48.244571
1,Panasonic Industrial Devices Materials Europe ...,AT,9.31,48.231083,14.496667,"POLYGON ((14.51685 48.21759, 14.51686 48.24457...",14.476471,48.217591,14.516863,48.244571
2,Imerys Villach GmbH,AT,2.18,46.620528,13.876833,"POLYGON ((13.89641 46.60703, 13.89642 46.63402...",13.857244,46.607033,13.896422,46.634020
3,Imerys Fused Minerals Villach GmbH,AT,2.18,46.620530,13.876833,"POLYGON ((13.89641 46.60703, 13.89642 46.63402...",13.857244,46.607035,13.896422,46.634022
4,Imerys Fused Minerals Villach GmbH,AT,2.18,46.620530,13.876833,"POLYGON ((13.89641 46.60703, 13.89642 46.63402...",13.857244,46.607035,13.896422,46.634022


## 🌍 4. STAC & Dask Processing

**Note:** Ensure you set the desired date range below.

1. **Setup:** Suppress warnings, load libraries (STAC, Dask, rioxarray).
2. **Config:** Set STAC API (`https://stac.eodc.eu/api/v1`), Collection (`GFM`), and Time Range.
3. **Directories:** Create output folders (`flood`, `exclusion`, `likelihood`).
4. **Deduplication:** Prepare `bboxes_gdf` and handle duplicate facility names.
5. **Worker Function (`worker_compute`):**
   - Queries STAC for one AOI.
   - Loads bands: `ensemble_flood_extent`, `exclusion_mask`, `ensemble_likelihood`.
   - Computes aggregated flood extent (max over time) and valid monthly masks.
6. **Orchestration:**
   - Runs in parallel (Dask Cluster or Local Threads).
   - Saves results as GeoTIFFs.
   - Logs progress to `results.csv`.


In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

# STAC / processing libs (must exist on workers)
import odc.stac, pystac, rioxarray
from pystac_client import Client as StacClient

# dask (optional)
try:
    from dask.distributed import get_client, as_completed as dask_as_completed
except Exception:
    get_client = None
    dask_as_completed = None

# CONFIG
API_URL = "https://stac.eodc.eu/api/v1"
COLLECTION_ID = "GFM"
TIME_RANGE = ("2021-01-01", "2021-12-30")

time_range_str = f"{TIME_RANGE[0]}_{TIME_RANGE[1]}"
BASE_OUT = os.path.join(os.getcwd(), "outputs", time_range_str)
os.makedirs(BASE_OUT, exist_ok=True)
OUT_FLOOD = os.path.join(BASE_OUT, "flood"); os.makedirs(OUT_FLOOD, exist_ok=True)
OUT_EXCLUSION = os.path.join(BASE_OUT, "exclusion"); os.makedirs(OUT_EXCLUSION, exist_ok=True)
OUT_LIKELIHOOD = os.path.join(BASE_OUT, "likelihood"); os.makedirs(OUT_LIKELIHOOD, exist_ok=True)
RESULTS_CSV = os.path.join(BASE_OUT, "results.csv")

DASK_CHUNKS = {"x": 64, "y": 64, "time": 32}
BANDS = ["ensemble_flood_extent", "exclusion_mask", "ensemble_likelihood"]

CONCURRENCY = 8
DESIRED_WORKERS = 12
PER_SUBMIT_SLEEP = 0.02
MAX_INFLIGHT = CONCURRENCY * 4

# INPUT
assert 'bboxes_gdf' in globals() and isinstance(bboxes_gdf, gpd.GeoDataFrame), "Missing bboxes_gdf!"

bboxes_gdf = bboxes_gdf.copy()
bboxes_gdf["facilityname"] = bboxes_gdf["facilityname"].astype(str).str.strip()
dup = bboxes_gdf["facilityname"].duplicated(keep=False)
bboxes_gdf.loc[dup, "facilityname"] = (
    bboxes_gdf.loc[dup, "facilityname"] + "_" +
    bboxes_gdf.loc[dup].groupby("facilityname").cumcount().astype(str)
)
bboxes_gdf = bboxes_gdf[bboxes_gdf.geometry.notnull() & bboxes_gdf.geometry.is_valid & ~bboxes_gdf.geometry.is_empty]
AOIS = list(zip(bboxes_gdf["facilityname"].tolist(), bboxes_gdf.geometry.values.tolist()))
print("🗺️ Prepared", len(AOIS), "AOIs")

def slugify(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s).strip("_")

def outputs_exist(name: str) -> bool:
    safe = slugify(name)
    pref = f"{safe}_"
    try:
        for fn in os.listdir(OUT_FLOOD):
            if fn.startswith(pref) and fn.endswith(".tif"):
                return True
    except FileNotFoundError:
        return False
    return False

# WORKER
def worker_compute(name: str, aoi_geojson, aoi_bounds, api_url=API_URL,
                   collection_id=COLLECTION_ID, time_start=TIME_RANGE[0], time_end=TIME_RANGE[1]) -> dict:
    import pandas as _pd, pyproj as _pyproj, xarray as _xr, numpy as _np, socket as _socket
    import odc.stac as _odc
    from pystac_client import Client as _Client
    import dask, traceback as _tb

    row = {"name": name, "status": "error", "n_items": 0,
           "flood": None, "excl": None, "lik": None,
           "time_vals": [], "flood_months": [],
           "host": _socket.gethostname(), "error": "", "trace": ""}

    try:
        cat = _Client.open(api_url)
        a = _pd.to_datetime(time_start); b = _pd.to_datetime(time_end)
        edges = list(_pd.date_range(a.normalize().replace(day=1), b, freq="MS"))
        if not edges or edges[0] > a: edges = [a] + edges
        if edges[-1] < b: edges = edges + [b]

        items_all = []
        for aa, bb in zip(edges[:-1], edges[1:]):
            dt = f"{aa.strftime('%Y-%m-%dT%H:%M:%SZ')}/{bb.strftime('%Y-%m-%dT%H:%M:%SZ')}"
            s = cat.search(collections=[collection_id], intersects=aoi_geojson, datetime=dt)
            items = list(s.get_all_items()) if hasattr(s, "get_all_items") else list(s.item_collection() or [])
            if items: items_all.extend(items)

        if not items_all:
            row["status"] = "no_items"; return row
        row["n_items"] = len(items_all)

        p = items_all[0].properties
        if "proj:wkt2" in p: crs = _pyproj.CRS.from_wkt(p["proj:wkt2"])
        elif p.get("proj:epsg"): crs = _pyproj.CRS.from_epsg(int(p["proj:epsg"]))
        else: crs = _pyproj.CRS.from_epsg(4326)
        gsd = p.get("gsd")
        res = float(gsd[0]) if isinstance(gsd,(list,tuple)) and gsd else (float(gsd) if isinstance(gsd,(int,float)) else None)

        xx = _odc.load(items_all, bbox=aoi_bounds, crs=crs, bands=BANDS,
                       resolution=res, dtype="uint8", chunks=DASK_CHUNKS,
                       skip_broken_datasets=True,
                       resampling={b:"nearest" for b in BANDS})
        if "time" in xx.dims: xx = xx.sortby("time")

        data = xx.ensemble_flood_extent
        valid = (data != 255) & (data != 0)
        flood_da = valid.any(dim="time").astype("uint8")
        flood_time_mask = valid.any(dim=("y","x"))  # bool per time
        time_da = data.time

        # select last available exclusion raster (non-255) — more efficient than taking isel(time=-1) blindly
        excl_da = None
        if "exclusion_mask" in xx.data_vars and xx.sizes.get("time", 0) > 0:
            excl_raw = xx.exclusion_mask
            # replace 255 with NaN to allow testing which time steps have at least one valid pixel
            excl_valid = excl_raw.where(excl_raw != 255)

            # boolean for each time, whether it has at least one valid pixel
            any_valid = (~excl_valid.isnull()).any(dim=("y", "x"))

            if any_valid.any():
                # find the last index with at least one valid pixel
                valid_idxs = _np.flatnonzero(any_valid.values)
                last_idx = int(valid_idxs[-1])
                excl_last = excl_valid.isel(time=last_idx)
                # convert to 0/1 uint8 (1 = excluded)
                excl_da = (excl_last == 1).astype("uint8")
            else:
                # no valid exclusion raster -> leave None (will not be written)
                excl_da = None

        if "ensemble_likelihood" in xx.data_vars and xx.sizes.get("time",0)>0:
            lik_raw = xx["ensemble_likelihood"].where(xx["ensemble_likelihood"]!=255)
            valid_dom = (xx.ensemble_flood_extent!=255).any(dim="time")
            lik_da = lik_raw.where(valid_dom).max(dim="time",skipna=True).fillna(255).astype("uint8")

        import dask
        res_tuple = dask.compute(
            flood_da, flood_time_mask, time_da,
            *( [excl_da] if excl_da is not None else [] ),
            *( [lik_da] if lik_da is not None else [] )
        )
        idx = 0
        flood_c, mask_c, time_vals_c = res_tuple[0:3]; idx = 3
        excl_c = res_tuple[idx] if excl_da is not None else None; idx += (1 if excl_da is not None else 0)
        lik_c  = res_tuple[idx] if lik_da is not None else None

        # months with flood
        tv = _pd.to_datetime(_np.asarray(time_vals_c.values))
        mk = _np.asarray(mask_c).astype(bool)
        flood_months = (sorted(_pd.unique(_pd.to_datetime(tv[mk]).to_period("M").astype(str)))
                        if tv.size and mk.any() else [])

        row.update({
            "status":"ok",
            "flood":flood_c, "excl":excl_c, "lik":lik_c,
            "time_vals": list(tv.astype("datetime64[ns]")),
            "flood_months": flood_months,
        })
        return row

    except Exception as e:
        row["status"]="error"; row["error"]=repr(e); row["trace"]=_tb.format_exc()
        return row

# DRIVER: SAVE
def save_result(res: dict):
    if res.get("status")!="ok": return
    safe = slugify(res["name"])
    months = res.get("flood_months") or []
    flood_suffix = "_".join(months) if months else "no_flood"

    flood_path = os.path.join(OUT_FLOOD,f"{safe}_max_flood_{flood_suffix}.tif")
    res["flood"].rio.to_raster(flood_path,compress="LZW",tiled=True,blockxsize=256,blockysize=256,BIGTIFF="IF_SAFER")

    time_vals = pd.to_datetime(res.get("time_vals", []))
    if res.get("excl") is not None and time_vals.size:
        excl_suffix = pd.to_datetime(time_vals[-1]).to_period("M").strftime("%Y-%m")
        excl_path = os.path.join(OUT_EXCLUSION,f"{safe}_exclusion_{excl_suffix}.tif")
        res["excl"].rio.to_raster(excl_path,compress="LZW",tiled=True,blockxsize=256,blockysize=256,BIGTIFF="IF_SAFER")

    if res.get("lik") is not None:
        lik_path = os.path.join(OUT_LIKELIHOOD,f"{safe}_ensemble_likelihood_max.tif")
        res["lik"].rio.to_raster(lik_path,compress="LZW",tiled=True,blockxsize=256,blockysize=256,BIGTIFF="IF_SAFER")

def _append_row_csv(row:dict):
    header = not os.path.exists(RESULTS_CSV)
    pd.DataFrame([row]).to_csv(RESULTS_CSV,mode="a",header=header,index=False)

# DRIVER ORCHESTRATION (STREAM SAVE)
to_process=[(n,g) for n,g in AOIS if not outputs_exist(n)]
print(f"Total AOIs: {len(AOIS)} | To process: {len(to_process)} | Skipped: {len(AOIS)-len(to_process)}")

client=None; use_cluster=False
try:
    if get_client: client=get_client(); use_cluster=client is not None
except Exception:
    pass

if use_cluster:
    print("Dask client detected — running on cluster.")
    try:
        if DESIRED_WORKERS and hasattr(client, "cluster"):
            client.cluster.scale(DESIRED_WORKERS)
    except Exception:
        pass

    from dask.distributed import as_completed as _dask_ac
    futures=[]; fut2name={}; idx=0; total=len(to_process)

    while idx < total and len(futures) < MAX_INFLIGHT:
        n,g = to_process[idx]
        f = client.submit(worker_compute, n, g.__geo_interface__, g.bounds, pure=False)
        futures.append(f); fut2name[f]=n; idx += 1; time.sleep(PER_SUBMIT_SLEEP)

    ac = _dask_ac(futures)
    pbar = tqdm(total=total, desc="Processing AOIs (cluster)", unit="AOI")

    for done in ac:
        try:
            res = done.result()
            save_result(res); _append_row_csv(res)
            print(f"  {res.get('name','?')}: {res.get('status')} n={res.get('n_items')} months={','.join(res.get('flood_months',[])) or 'no_flood'}")
        except Exception as e:
            err = {"name": fut2name.get(done,"?"), "status":"error", "error":repr(e), "host":"driver", "trace":traceback.format_exc()}
            _append_row_csv(err); print(f"  {err['name']} ERROR: {e!r}")
        pbar.update(1)

        if idx < total:
            n,g = to_process[idx]
            nf = client.submit(worker_compute, n, g.__geo_interface__, g.bounds, pure=False)
            ac.add(nf); fut2name[nf]=n; idx += 1; time.sleep(PER_SUBMIT_SLEEP)
    pbar.close()

else:
    print("No cluster — local threads.")
    with ThreadPoolExecutor(max_workers=CONCURRENCY) as ex:
        fut2name = {ex.submit(worker_compute, n, g.__geo_interface__, g.bounds): n
                    for n, g in to_process}
        for fut in tqdm(as_completed(fut2name), total=len(fut2name), desc="Processing AOIs (local)", unit="AOI"):
            try:
                res = fut.result()
                save_result(res); _append_row_csv(res)
                print(f"  {res['name']}: {res['status']} n={res['n_items']} months={','.join(res.get('flood_months',[])) or 'no_flood'}")
            except Exception as e:
                err = {"name": fut2name[fut], "status":"error", "error":repr(e), "host":socket.gethostname(), "trace":traceback.format_exc()}
                _append_row_csv(err); print(f"  {fut2name[fut]} ERROR: {e!r}")

print("DONE. Outputs at:", BASE_OUT)
print("  Flood:", OUT_FLOOD)
print("  Exclusion:", OUT_EXCLUSION)
print("  Likelihood:", OUT_LIKELIHOOD)
print("Log CSV ->", RESULTS_CSV)


🗺️ Prepared 1505 AOIs
Total AOIs: 1505 | To process: 1505 | Skipped: 0
Dask client detected — running on cluster.


Processing AOIs (cluster):   0%|          | 0/1505 [00:00<?, ?AOI/s]

  Wienerberger Ltd: ok n=820 months=2021-01,2021-02,2021-04,2021-05,2021-06,2021-07
  Sandtoft Roof Tiles Ltd: ok n=820 months=2021-01,2021-02,2021-04,2021-05,2021-06,2021-07
  WIENERBERGER SAS_0: ok n=936 months=2021-03,2021-04
  Imerys Fused Minerals Villach GmbH_1: ok n=943 months=2021-01,2021-02
  Imerys Fused Minerals Villach GmbH_0: ok n=943 months=2021-01,2021-02
  Imerys Villach GmbH: ok n=943 months=2021-01,2021-02
  VERBUND Thermal Power GmbH & Co KG in Liqu._0: ok n=991 months=no_flood
  Panasonic Industrial Devices Materials Europe GmbH_0 ERROR: FutureCancelledError()
  Panasonic Industrial Devices Materials Europe GmbH_1 ERROR: FutureCancelledError()
  WIENERBERGER_0 ERROR: FutureCancelledError()
  WIENERBERGER_1 ERROR: FutureCancelledError()
  WIENERBERGER_2 ERROR: FutureCancelledError()
  Wienerberger T�glaipari Zrt. ERROR: FutureCancelledError()
  WIENERBERGER Zrt._0 ERROR: FutureCancelledError()
  WIENERBERGER Zrt._1 ERROR: FutureCancelledError()
  voestalpine Wire Aus

ClosedClientError: Client is closed. Can't send update-graph message.

## 📊 5. Raster Statistics Calculation

1. **Config:** Set output paths and Projection (`EPSG:3035` - Equal Area).
2. **Regex:** Identify file types (flood/exclusion/likelihood) from filenames.
3. **Helpers:** Functions for calculating pixel area, centroids, and reprojection.
4. **Calculator (`stats_for_file`):**
   - Loads raster, counts valid pixels.
   - Computes flooded percentage or likelihood stats.
5. **Batch Processing:** Iterates over all `.tif` files in output directories.
6. **Export:**
   - `area_by_raster.csv`: Detailed stats per file.
   - `area_by_facility.csv`: Summary per facility.


In [ ]:
# Configuration
BASE_DIR = Path(BASE_OUT).resolve()
dirs = {
    "flood": BASE_DIR / "flood",
    "exclusion": BASE_DIR / "exclusion",
    "likelihood": BASE_DIR / "likelihood",
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

OUT_PER_FILE = BASE_DIR / "area_by_raster.csv"
OUT_PER_FAC  = BASE_DIR / "area_by_facility.csv"
EQUAL_AREA_EPSG = "EPSG:3035"  # ETRS89 / LAEA Europe

# filename regexes
pat_flood = re.compile(r"^(?P<name>.+?)_max_flood_(?P<suffix>.+)\.tif$", re.IGNORECASE)
pat_excl  = re.compile(r"^(?P<name>.+?)_exclusion_(?P<suffix>.+)\.tif$", re.IGNORECASE)
pat_lik   = re.compile(r"^(?P<name>.+?)_ensemble_likelihood_(?P<suffix>.+)\.tif$", re.IGNORECASE)

def is_degree_crs(crs):
    try:
        s = crs.to_string().lower()
    except Exception:
        s = str(crs or "").lower()
    return ("longlat" in s) or ("deg" in s)

def pixel_area_m2(da):
    rx, ry = da.rio.resolution()
    return abs(float(rx) * float(ry))

def centroid_lonlat(da):
    try:
        minx, miny, maxx, maxy = da.rio.bounds()
        cx = (minx + maxx) / 2.0
        cy = (miny + maxy) / 2.0
        tr = Transformer.from_crs(da.rio.crs, "EPSG:4326", always_xy=True)
        lon, lat = tr.transform(cx, cy)
        return float(lon), float(lat)
    except Exception:
        return None, None

def open_da_equal_area(path: Path, kind: str):
    da = rxr.open_rasterio(str(path), masked=True).squeeze()
    if "band" in da.dims:
        da = da.isel(band=0).drop_vars("band", errors="ignore")
    da = da.astype("float32")
    if kind in ("flood","exclusion"):
        da = xr.where(da == 1, 1, 0)
    else:  # likelihood: keep values, 255 = NoData
        da = da.where(da != 255)
    if is_degree_crs(da.rio.crs):
        da = da.rio.reproject(EQUAL_AREA_EPSG, resampling=Resampling.nearest)
    return da.load()

def stats_for_file(path: Path):
    fname = path.name
    kind, name, suffix = None, None, None
    m = pat_flood.match(fname)
    if m:
        kind, name, suffix = "flood", m.group("name"), m.group("suffix")
    else:
        m = pat_excl.match(fname)
        if m:
            kind, name, suffix = "exclusion", m.group("name"), m.group("suffix")
        else:
            m = pat_lik.match(fname)
            if m:
                kind, name, suffix = "likelihood", m.group("name"), m.group("suffix")

    if kind is None:
        return None

    try:
        da = open_da_equal_area(path, kind)
        lon, lat = centroid_lonlat(da)
    except Exception as e:
        return {"file": fname, "facility": name, "kind": kind,
                "status": f"open_error: {e}", "path": str(path)}

    try:
        valid_mask = np.isfinite(da.values)
        n_valid = int(valid_mask.sum())
        if n_valid == 0:
            return {"file": fname, "facility": name, "kind": kind,
                    "suffix": suffix, "status": "no_valid_pixels", "path": str(path)}

        a_pix = pixel_area_m2(da)

        if kind in ("flood","exclusion"):
            n_ones = int((da.values == 1).sum())
            area_valid_m2 = n_valid * a_pix
            area_ones_m2  = n_ones * a_pix
            pct = round((area_ones_m2 / area_valid_m2) * 100.0, 0)
            present = int(n_ones > 0)
            lik_mean = lik_min = lik_max = None
        else:  # likelihood
            vals = da.values[valid_mask]
            if vals.size > 0:
                lik_mean = float(np.nanmean(vals))
                lik_min  = float(np.nanmin(vals))
                lik_max  = float(np.nanmax(vals))
                pct = round(lik_mean, 0)  # percentage probability
                present = int(lik_max > 0)
                n_ones = None
                area_valid_m2 = n_valid * a_pix
                area_ones_m2  = None
            else:
                lik_mean=lik_min=lik_max=None
                pct=0.0; present=0; n_ones=None; area_valid_m2=0; area_ones_m2=None

        return {
            "file": fname,
            "facility": name,
            "kind": kind,
            "suffix": suffix,
            "status": "ok",
            "n_pixels_valid": n_valid,
            "n_pixels_1": n_ones,
            "pixel_area_m2": a_pix,
            "area_valid_m2": area_valid_m2,
            "area_1_m2": area_ones_m2,
            "pct_area_1": pct,
            "likelihood_mean": lik_mean,
            "likelihood_min": lik_min,
            "likelihood_max": lik_max,
            "present": present,
            "path": str(path),
            "crs": str(da.rio.crs),
            "lon": lon,
            "lat": lat,
        }
    except Exception as e:
        return {"file": fname, "facility": name, "kind": kind,
                "suffix": suffix, "status": f"calc_error: {e}", "path": str(path)}

# list tifs
tifs = []
for k, d in dirs.items():
    tifs.extend(glob.glob(str(d / "*.tif")))
    tifs.extend(glob.glob(str(d / "*.TIF")))
print(f"Found {len(tifs)} GeoTIFF(s) in subdirs of {BASE_DIR}")

rows = []
start = time.time()
with tqdm(total=len(tifs), desc="Processing rasters", unit="tif") as pbar:
    for p in tifs:
        res = stats_for_file(Path(p))
        if res is not None:
            rows.append(res)
        ok = sum(1 for r in rows if r.get("status") == "ok")
        bad = sum(1 for r in rows if r.get("status") != "ok")
        pbar.set_postfix({"ok": ok, "bad": bad}, refresh=True)
        pbar.update(1)

elapsed = time.time() - start

df_files = pd.DataFrame(rows)
if not df_files.empty:
    df_files.to_csv(OUT_PER_FILE, index=False)
    print(f"Saved per-file → {OUT_PER_FILE}")

    ok_df = df_files[df_files["status"] == "ok"].copy()

    facilities = []
    for fac, sub in ok_df.groupby("facility"):
        row = {"facility": fac}
        # Flood
        sub_flood = sub[sub["kind"] == "flood"]
        if not sub_flood.empty:
            row["flood_present"] = int((sub_flood["pct_area_1"] > 0).any())
            row["flood_pct"] = round(float(sub_flood["pct_area_1"].max()), 0)
            months = [s for s in sub_flood["suffix"].unique() if s != "no_flood"]
            row["flood_months"] = ",".join(sorted(months)) if months else ""
        else:
            row.update({"flood_present":0,"flood_pct":0.0,"flood_months":""})
        # Exclusion
        sub_excl = sub[sub["kind"] == "exclusion"]
        if not sub_excl.empty:
            row["exclusion_present"] = int((sub_excl["pct_area_1"] > 0).any())
            row["exclusion_pct"] = round(float(sub_excl["pct_area_1"].max()), 0)
            row["exclusion_suffix"] = ",".join(sorted(sub_excl["suffix"].unique()))
        else:
            row.update({"exclusion_present":0,"exclusion_pct":0.0,"exclusion_suffix":""})
        # Likelihood
        sub_lik = sub[sub["kind"] == "likelihood"]
        if not sub_lik.empty:
            row["likelihood_present"] = int((sub_lik["likelihood_max"]>0).any())
            row["likelihood_mean"]    = round(float(sub_lik["likelihood_mean"].mean()),1)
            row["likelihood_max"]     = float(sub_lik["likelihood_max"].max())
        else:
            row.update({"likelihood_present":0,"likelihood_mean":0.0,"likelihood_max":0.0})
        # Lon/Lat – median from rasters for facility
        if "lon" in sub.columns and sub["lon"].notna().any():
            row["lon"] = round(float(sub["lon"].median()), 6)
            row["lat"] = round(float(sub["lat"].median()), 6)
        else:
            row["lon"] = None
            row["lat"] = None

        facilities.append(row)

    df_fac = pd.DataFrame(facilities)
    df_fac.to_csv(OUT_PER_FAC, index=False)
    print(f"Saved per-facility → {OUT_PER_FAC}")
    display(df_fac.head(10))
else:
    print("No rasters parsed.")

print(f"Elapsed: {elapsed/60:.2f} min ({elapsed:.1f} s)")


Found 21 GeoTIFF(s) in subdirs of /Users/martinjancovic/Desktop/Szilard/outputs/2021-01-01_2021-12-30


Processing rasters:   0%|          | 0/21 [00:00<?, ?tif/s]

Saved per-file → /Users/martinjancovic/Desktop/Szilard/outputs/2021-01-01_2021-12-30/area_by_raster.csv
Saved per-facility → /Users/martinjancovic/Desktop/Szilard/outputs/2021-01-01_2021-12-30/area_by_facility.csv


,facility,flood_present,flood_pct,flood_months,exclusion_present,exclusion_pct,exclusion_suffix,likelihood_present,likelihood_mean,likelihood_max,lon,lat
0,Imerys_Fused_Minerals_Villach_GmbH_0,1,5.0,2021-01_2021-02,1,60.0,2021-12,1,21.0,100.0,13.876886,46.620572
1,Imerys_Fused_Minerals_Villach_GmbH_1,1,5.0,2021-01_2021-02,1,60.0,2021-12,1,21.0,100.0,13.876886,46.620572
2,Imerys_Villach_GmbH,1,5.0,2021-01_2021-02,1,60.0,2021-12,1,21.0,100.0,13.876886,46.620572
3,Sandtoft_Roof_Tiles_Ltd,1,18.0,2021-01_2021-02_2021-04_2021-05_2021-06_2021-07,1,7.0,2021-12,1,43.4,100.0,-0.698808,53.736636
4,VERBUND_Thermal_Power_GmbH_Co_KG_in_Liqu._0,0,0.0,,1,48.0,2021-12,1,17.1,49.0,15.483580,46.908859
5,WIENERBERGER_SAS_0,1,1.0,2021-03_2021-04,1,38.0,2021-12,1,23.9,99.0,4.930432,44.010110
6,Wienerberger_Ltd,1,18.0,2021-01_2021-02_2021-04_2021-05_2021-06_2021-07,1,7.0,2021-12,1,43.4,100.0,-0.698808,53.736636


Elapsed: 0.03 min (1.9 s)


## 💧 6. Alert Matching (HEFAS)

1. **Input:** Load facility locations from `area_by_facility.csv`.
2. **Geometry:** Create GeoDataFrame (EPSG:3035).
3. **Timeline Loop:** For each day in the study period:
   - Fetch HEFAS alerts (WFS) for that date.
   - Calculate distance to mapped facilities (within 100km).
   - metrics: `flood_any`, `level_mean`, `min_dist_m`.
4. **Aggregation:**
   - **Per-Month:** Which months had active alerts.
   - **Per-Facility:** Summary stats (days with flood, max level).
5. **Final Output:** `facility_with_floodstats_summary.csv`.

For credentials contact rafael.garcia@soologic.com


In [ ]:
CSV_PATH = os.path.join(BASE_OUT, "area_by_facility.csv")
N_NEAREST = 3
MAX_DIST_KM = 100.0  # stations farther than this are ignored

#  FETCH ALERTS: UNION OF TWO LAYERS (NationalThresholdExceedance + PegelOnline)
def fetch_alerts(date_str):
    base_url = "https://ehdcc.soologic.com/geoserver/HEFAS/ows"
    auth = HTTPBasicAuth("user_name", "password")

    gdfs = []

    # LAYER 1: NationalThresholdExceedance
    params1 = {
        "service": "WFS",
        "version": "1.0.0",
        "request": "GetFeature",
        "typeName": "HEFAS:NationalThresholdExceedance",
        "viewparams": f"date_Search:{date_str}",
        "outputFormat": "application/json",
    }

    r1 = requests.get(base_url, params=params1, auth=auth, timeout=30)
    r1.raise_for_status()
    g1 = gpd.read_file(BytesIO(r1.content))

    if not g1.empty:
        g1["Level_Exceeded"] = pd.to_numeric(g1.get("Level_Exceeded", 0), errors="coerce").fillna(0)
        gdfs.append(g1)

    # LAYER 2: PegelOnline (Germany)
    params2 = {
        "service": "WFS",
        "version": "1.0.0",
        "request": "GetFeature",
        "typeName": "HEFAS:PegelOnline",
        "viewparams": f"date_Search:{date_str}",
        "outputFormat": "application/json",
    }

    r2 = requests.get(base_url, params=params2, auth=auth, timeout=30)
    r2.raise_for_status()
    g2 = gpd.read_file(BytesIO(r2.content))

    if not g2.empty:
        # if Pegel is missing Level_Exceeded -> set to 0
        if "Level_Exceeded" in g2.columns:
            g2["Level_Exceeded"] = pd.to_numeric(g2["Level_Exceeded"], errors="coerce").fillna(0)
        else:
            g2["Level_Exceeded"] = 0
        gdfs.append(g2)

    # UNION OF BOTH
    if not gdfs:
        return gpd.GeoDataFrame(columns=["Level_Exceeded", "geometry"], geometry="geometry", crs="EPSG:4326")

    combined = pd.concat(gdfs, ignore_index=True)
    if combined.crs is None:
        combined = combined.set_crs("EPSG:4326")

    return combined


#  LOAD FACILITIES CSV (no changes!)
df = pd.read_csv(CSV_PATH)

required = {"facility", "lon", "lat"}
missing = required - set(df.columns)
if missing:
    raise ValueError(
        f"CSV must contain: {sorted(required)}. Got: {list(df.columns)}"
    )

points_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:3035")



#  MAIN LOOP
dates = pd.date_range(*TIME_RANGE, freq="D")
max_dist_m = MAX_DIST_KM * 1000.0

records = []
for d in tqdm(dates, desc="days"):
    gdf_alerts = fetch_alerts(d.strftime("%Y-%m-%d"))
    if gdf_alerts is None or gdf_alerts.empty:
        continue

    alerts_3035 = gdf_alerts.to_crs("EPSG:3035")

    for idx, pt in points_gdf.iterrows():
        dists = alerts_3035.geometry.distance(pt.geometry)

        if dists.empty:
            records.append({
                "date_req": d, "facility": pt["facility"],
                "flood_any": False, "level_mean": 0.0, "level_max": 0.0,
                "min_dist_m": None, "n_neighbors": 0
            })
            continue

        ordered = dists.sort_values()
        within = ordered[ordered <= max_dist_m]

        if within.empty:
            records.append({
                "date_req": d, "facility": pt["facility"],
                "flood_any": False, "level_mean": 0.0, "level_max": 0.0,
                "min_dist_m": None, "n_neighbors": 0
            })
            continue

        nearest_idx = within.index[:N_NEAREST]
        nearest = alerts_3035.loc[nearest_idx]
        nearest_dists = within.loc[nearest_idx]

        level_values = pd.to_numeric(nearest["Level_Exceeded"], errors="coerce").fillna(0).values
        min_dist = float(nearest_dists.min())

        flood_any = (level_values > 0).any()
        flood_majority = (level_values > 0).sum() >= (len(level_values)//2 + 1)
        weights = 1.0 / (nearest_dists.values + 1e-6)
        weighted_score = (level_values * weights).sum() / weights.sum() if weights.sum() > 0 else 0.0
        flood_weighted = weighted_score > 0

        records.append({
            "date_req": d,
            "facility": pt["facility"],
            "flood_any": bool(flood_any),
            "flood_majority": bool(flood_majority),
            "flood_weighted": bool(flood_weighted),
            "level_mean": float(level_values.mean()),
            "level_max": float(level_values.max()),
            "min_dist_m": min_dist,
            "n_neighbors": int(len(level_values))
        })



#  AGGREGATION
if records:
    df_all = pd.DataFrame(records)
    df_all["month"] = df_all["date_req"].dt.to_period("M").astype(str)

    per_month = (
        df_all.groupby(["facility", "month"])
        .agg(
            any_flood_any=("flood_any", "max"),
            flood_days_any=("flood_any", "sum"),
            level_mean=("level_mean", "mean"),
            level_max=("level_max", "max"),
            min_dist_m=("min_dist_m", "min")
        )
        .reset_index()
    )

    def collect_months(sub):
        months = sorted(sub.loc[sub["any_flood_any"] == 1, "month"].unique())
        return ",".join(months)

    per_fac = (
        per_month.groupby("facility")
        .agg(
            HEFAS_months=("month", lambda s: collect_months(per_month.loc[per_month["facility"] == s.name])),
            HEFAS_flood_days_any=("flood_days_any", "sum"),
            HEFAS_any_flood_any=("any_flood_any", "max"),
            HEFAS_level_mean=("level_mean", "mean"),
            HEFAS_level_max=("level_max", "max"),
            HEFAS_min_dist_km=("min_dist_m", lambda v: None if v.isnull().all() else round(v.min() / 1000.0, 1))
        )
        .reset_index()
    )

    merged = df.merge(per_fac, on="facility", how="left")
    merged.to_csv(os.path.join(BASE_OUT, "facility_with_floodstats_summary.csv"), index=False)
    print("Saved facility_with_floodstats_summary.csv")

else:
    print("No alerts in period.")


days:   0%|          | 0/364 [00:00<?, ?it/s]

Saved facility_with_floodstats_summary.csv
